<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day2/notebooks/3_embeddings_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 2 — Word Embeddings in Practice  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every **✏️ Exercise** is filled in with one possible
answer plus a short comment. Because embeddings depend on the training corpus, your exact
numbers and neighbors may differ slightly.


## 0. Setup

In [ ]:
!pip install gensim --q

import numpy as np
import matplotlib.pyplot as plt

import gensim.downloader as api
from gensim.models import Word2Vec

print("Setup complete!")

## 1. Loading pre-trained embeddings

In [ ]:
glove = api.load("glove-wiki-gigaword-50")
print("Loaded!")
print("Vocabulary size:", len(glove))
print("Vector size (dimensions):", glove.vector_size)

In [ ]:
vec = glove["king"]
print("The vector for 'king' has", len(vec), "dimensions.")
print()
print("First 10 numbers:")
print(vec[:10])

> **✏️ Exercise 1**
>
> Pull out the vector for a word of your choice. Confirm it has the same length as ``king``.


In [ ]:
# ✅ Solution
word = "democracy"
vec = glove[word]
print(f"Vector for '{word}' has {len(vec)} dimensions.")
print("Same length as 'king'?", len(vec) == len(glove["king"]))

# Comment: every word in a given embedding model has a vector of the SAME length
# (here 50). That fixed size is what lets us compare, add, and average vectors.

## 2. Similarity and nearest neighbors

In [ ]:
print("king  vs queen: ", round(glove.similarity("king", "queen"), 3))
print("king  vs man:   ", round(glove.similarity("king", "man"), 3))
print("king  vs banana:", round(glove.similarity("king", "banana"), 3))

In [ ]:
for word, score in glove.most_similar("university", topn=10):
    print(f"  {word:<15} {score:.3f}")

> **✏️ Exercise 2**
>
> Find the 10 nearest neighbors of a word relevant to your research. Do they make sense?


In [ ]:
# ✅ Solution
for w in ["policy", "media", "health"]:
    print(f"Nearest neighbors of '{w}':")
    for word, score in glove.most_similar(w, topn=5):
        print(f"  {word:<15} {score:.3f}")
    print()

# Comment: the neighbors are mostly sensible ("policy" -> "policies", "reform",
# "economic"...). Note they mix true synonyms, morphological variants (plurals),
# and topically-related words. Embeddings capture "relatedness", which is broader
# than strict synonymy — worth remembering when you interpret them.

## 3. Word analogies

In [ ]:
result = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print("king - man + woman ≈")
for word, score in result:
    print(f"  {word:<12} {score:.3f}")

In [ ]:
result = glove.most_similar(positive=["paris", "germany"], negative=["france"], topn=3)
print("paris - france + germany ≈")
for word, score in result:
    print(f"  {word:<12} {score:.3f}")

> **✏️ Exercise 3**
>
> Try your own analogy, and note one that fails.


In [ ]:
# ✅ Solution
# A working one: comparative adjectives
print("good - better + bad ≈")
for word, score in glove.most_similar(positive=["better", "bad"], negative=["good"], topn=3):
    print(f"  {word:<12} {score:.3f}")
print()

# One that often DISAPPOINTS: professions / gendered analogies can be noisy or biased
print("doctor - man + woman ≈")
for word, score in glove.most_similar(positive=["doctor", "woman"], negative=["man"], topn=3):
    print(f"  {word:<12} {score:.3f}")

# Comment: "better - good + bad" gives "worse" nicely. The doctor analogy often
# returns "nurse" — not a logical error, but a reflection of gender STEREOTYPES in
# the training text. That is a preview of the "bias in embeddings" discussion:
# the geometry faithfully encodes patterns in the data, including the troubling ones.

## 4. Visualizing the embedding space

In [ ]:
from sklearn.decomposition import PCA

words = [
    "king", "queen", "prince", "princess",
    "dog", "cat", "horse", "cow",
    "paris", "london", "berlin", "tokyo",
    "happy", "joyful", "sad", "angry",
]
vectors = np.array([glove[w] for w in words])
coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(11, 8))
plt.scatter(coords[:, 0], coords[:, 1], color="#34B233", s=60)
for i, word in enumerate(words):
    plt.annotate(word, (coords[i, 0], coords[i, 1]),
                 fontsize=12, xytext=(5, 5), textcoords="offset points")
plt.title("Word embeddings projected to 2-D (PCA)")
plt.xlabel("Component 1"); plt.ylabel("Component 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **✏️ Exercise 4**
>
> Make your own plot with words from two or three categories you care about.


In [ ]:
# ✅ Solution
words = [
    "election", "vote", "democracy", "parliament",   # politics
    "disease", "patient", "hospital", "treatment",   # health
    "river", "mountain", "forest", "ocean",          # nature
]
vectors = np.array([glove[w] for w in words])
coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(11, 8))
plt.scatter(coords[:, 0], coords[:, 1], color="#1A1A2E", s=60)
for i, word in enumerate(words):
    plt.annotate(word, (coords[i, 0], coords[i, 1]),
                 fontsize=12, xytext=(5, 5), textcoords="offset points")
plt.title("Three domains in embedding space")
plt.xlabel("Component 1"); plt.ylabel("Component 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Comment: the three domains generally separate into distinct regions. But note some
# overlap and the odd misplaced word — 2-D PCA discards most of the 50 dimensions,
# so the picture is suggestive, not definitive.

## 5. Training your own embeddings

In [ ]:
import nltk
nltk.download("brown")
from nltk.corpus import brown

sentences = [[w.lower() for w in sent] for sent in brown.sents()]
print("Number of sentences:", len(sentences))
print("Example sentence:", sentences[0][:12], "...")

In [ ]:
model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,
    epochs=5,
    seed=42,
)
print("Training complete!")
print("Vocabulary size:", len(model.wv))

In [ ]:
word = "money"
if word in model.wv:
    print(f"Words most similar to '{word}' in our Brown-trained model:")
    for w, score in model.wv.most_similar(word, topn=8):
        print(f"  {w:<15} {score:.3f}")
else:
    print(f"'{word}' not in vocabulary — try another.")

> **✏️ Exercise 5**
>
> Compare the same word's nearest neighbors in `glove` vs your Brown-trained `model.wv`.


In [ ]:
# ✅ Solution
word = "school"

print("PRE-TRAINED GloVe (Wikipedia + news):")
for w, score in glove.most_similar(word, topn=8):
    print(f"  {w:<15} {score:.3f}")

print("\nOUR Brown-trained Word2Vec:")
for w, score in model.wv.most_similar(word, topn=8):
    print(f"  {w:<15} {score:.3f}")

# Comment: the GloVe neighbors are cleaner and more clearly school-related, because
# GloVe saw BILLIONS of words. The Brown neighbors are noisier — Brown has ~1 million
# words, far too few for high-quality embeddings. Lesson: training your own only pays
# off with a large, domain-relevant corpus; otherwise pre-trained wins.

> **✏️ Exercise 6**
>
> Re-train with CBOW (`sg=0`) and/or a larger window. Do the neighbors change?


In [ ]:
# ✅ Solution
model_cbow = Word2Vec(
    sentences,
    vector_size=100,
    window=10,     # larger window
    min_count=5,
    sg=0,          # CBOW
    epochs=5,
    seed=42,
)

word = "money"
print("Skip-gram (window=5) neighbors of 'money':")
print([w for w, s in model.wv.most_similar(word, topn=8)])
print()
print("CBOW (window=10) neighbors of 'money':")
print([w for w, s in model_cbow.wv.most_similar(word, topn=8)])

# Comment: the lists differ. A larger window tends to capture broader, more topical
# relatedness (words from the same subject area), while a small window leans toward
# tighter, more syntactic similarity. Neither is "right" — you tune to your question.

## 6. A short detour: measuring bias

Embeddings absorb the **social biases** in their training text. That is a hazard — but it
also lets us *measure* those biases. Here we build a small **gender-association score** using
the pre-trained `glove` vectors.


In [ ]:
male_words = ["he", "him", "his", "man", "male"]
female_words = ["she", "her", "hers", "woman", "female"]

def gender_association(word, model=glove):
    """Positive = leans male, negative = leans female (by cosine similarity)."""
    male_sim = np.mean([model.similarity(word, m) for m in male_words])
    female_sim = np.mean([model.similarity(word, f) for f in female_words])
    return male_sim - female_sim

print("Association score (positive = male-leaning, negative = female-leaning):")
print("  king: ", round(gender_association("king"), 3))
print("  queen:", round(gender_association("queen"), 3))

``king`` scores positive, ``queen`` negative — the measure behaves sensibly. Now
professions, which carry no inherent gender.


In [ ]:
professions = [
    "nurse", "teacher", "librarian", "receptionist",
    "engineer", "scientist", "programmer", "surgeon",
    "doctor", "lawyer", "professor", "assistant",
]
scores = [(p, gender_association(p)) for p in professions]
scores.sort(key=lambda x: x[1])

print(f"{'profession':<14}{'score':>8}   leaning")
print("-" * 40)
for prof, score in scores:
    lean = "male" if score > 0 else "female"
    print(f"{prof:<14}{score:>8.3f}   {lean}")

``nurse`` and ``receptionist`` lean female; ``engineer`` and ``programmer`` lean male —
the embedding absorbed occupational gender stereotypes from its training text.


In [ ]:
profs = [p for p, s in scores]
vals = [s for p, s in scores]
colors = ["#34B233" if v < 0 else "#1A1A2E" for v in vals]

plt.figure(figsize=(10, 6))
plt.barh(profs, vals, color=colors)
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Gender association of professions in GloVe\n(left = female-leaning, right = male-leaning)")
plt.xlabel("male-leaning  ->")
plt.tight_layout()
plt.show()

> **✏️ Exercise 7**
>
> Try it with your own words (e.g. adjectives) and think about the limitations of measuring
> bias this way.


In [ ]:
# ✅ Solution
adjectives = ["strong", "gentle", "ambitious", "caring", "aggressive", "nurturing",
              "rational", "emotional", "confident", "supportive"]

adj_scores = [(a, gender_association(a)) for a in adjectives]
adj_scores.sort(key=lambda x: x[1])
for adj, score in adj_scores:
    lean = "male" if score > 0 else "female"
    print(f"{adj:<12}{score:>8.3f}   {lean}")

# Comment on the LIMITATIONS (the important part):
# 1. The score depends entirely on our chosen anchor words — swap them and results shift.
# 2. A single embedding is one noisy sample; different corpora / runs give different numbers.
# 3. Cosine "association" is not the same as a validated social-science construct.
# 4. We have no uncertainty estimate — one number per word hides a lot.
# Before any research claim you would: test anchor robustness, train/compare multiple
# models, report variability, and validate against an external benchmark or human ratings.
# The measure is a starting point for inquiry, not evidence on its own.

## Wrap-up

That's the full solution set.

### Optional challenge

Average word vectors to get a sentence vector, then compare two sentences.


In [ ]:
# ✅ Solution
def sentence_vector(sentence, model=glove):
    """Average the embedding vectors of the words in a sentence."""
    words = sentence.lower().split()
    vectors = [model[w] for w in words if w in model]
    if not vectors:                       # no known words
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

def cosine(a, b):
    if np.all(a == 0) or np.all(b == 0):
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

s1 = "the government passed a new economic policy"
s2 = "lawmakers approved fresh financial legislation"   # similar meaning, different words
s3 = "the cat slept on the warm windowsill"             # unrelated

v1, v2, v3 = sentence_vector(s1), sentence_vector(s2), sentence_vector(s3)

print(f"s1 vs s2 (related):   {cosine(v1, v2):.3f}")
print(f"s1 vs s3 (unrelated): {cosine(v1, v3):.3f}")

# Comment: s1 and s2 score much higher than s1 and s3, even though s1 and s2 share
# almost NO words. This is exactly what bag-of-words could not do (recall the
# "car" vs "automobile" problem): averaged embeddings capture meaning beyond exact
# word overlap. Note the limitation too — averaging discards word order.